In [2]:
!pip install Node2Vec
!pip install lazypredict

In [2]:
import pandas as pd
import networkx as nx
from node2vec import Node2Vec
from sklearn.model_selection import train_test_split
from lazypredict.Supervised import LazyClassifier

In [3]:
#brew install wget
!pip install pycaret[full] python-louvain scikit-learn-intelex



zsh:1: no matches found: pycaret[full]


In [9]:
# Remove the wrong package if it exists, then install the right one
!pip uninstall -y community >/dev/null 2>&1
!pip install -q python-louvain

In [10]:
import time, math, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import networkx as nx
from scipy import sparse

# embeddings + scaling
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

# clustering
from sklearn.cluster import KMeans, DBSCAN
# Robust import regardless of environment quirks
try:
    # Preferred import path from python-louvain
    from community import community_louvain
except Exception:
    import community as community_louvain  # some envs expose it top-level

# Sanity check: should have best_partition
assert hasattr(community_louvain, "best_partition"), "python-louvain not installed correctly"


# clustering metrics
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# supervised sanity check (like you did)
from sklearn.model_selection import train_test_split
from lazypredict.Supervised import LazyClassifier


In [5]:
# ---- Yeast PPI ----
yeast_file = "yeast.edgelist"
G_yeast = nx.read_edgelist(yeast_file)

# ---- Human PPI ----
# Your CSV uses two columns that contain the endpoints (you already discovered these indices)
human_file = "PP-Pathways_ppi.csv"
df_h = pd.read_csv(human_file)
edges_human = list(zip(df_h['1394'].astype(str), df_h['2778'].astype(str)))
G_human = nx.Graph(); G_human.add_edges_from(edges_human)

print(f"Yeast: nodes={G_yeast.number_of_nodes():,} edges={G_yeast.number_of_edges():,}")
print(f"Human: nodes={G_human.number_of_nodes():,} edges={G_human.number_of_edges():,}")


Yeast: nodes=6,526 edges=532,180
Human: nodes=21,557 edges=342,352


In [6]:
def svd_embeddings(G, n_components=64, normalize_rows=True, standardize=True, random_state=42):
    t0 = time.time()
    # fixed node order for reproducibility
    nodes = np.array(list(G.nodes()))
    node_to_idx = {n:i for i,n in enumerate(nodes)}
    A = nx.to_scipy_sparse_array(G, nodelist=nodes, dtype=np.float32, format='csr')

    if normalize_rows:
        # row L1 normalize to keep degree effects controlled
        row_sums = np.asarray(A.sum(axis=1)).ravel()
        row_sums[row_sums==0] = 1.0
        Dinv = sparse.diags(1.0/row_sums)
        A_norm = Dinv @ A
    else:
        A_norm = A

    svd = TruncatedSVD(n_components=n_components, random_state=random_state)
    X = svd.fit_transform(A_norm)

    if standardize:
        X = StandardScaler(with_mean=False).fit_transform(X)  # with_mean=False safe for sparse pipeline notion

    elapsed = time.time() - t0
    print(f"SVD embeddings: n={G.number_of_nodes()}, d={n_components}, time={elapsed:.2f}s, explained_var_sum={svd.explained_variance_ratio_.sum():.4f}")
    return nodes, X

nodes_y, X_y = svd_embeddings(G_yeast, n_components=64)
nodes_h, X_h = svd_embeddings(G_human, n_components=64)


SVD embeddings: n=6526, d=64, time=0.65s, explained_var_sum=0.4859
SVD embeddings: n=21557, d=64, time=0.90s, explained_var_sum=0.3649


In [11]:
def evaluate_clustering(X, labels):
    """Compute clustering metrics. Ignores degenerate cases."""
    labels = np.asarray(labels)
    # Unique clusters excluding noise (-1) if present
    mask = labels != -1
    uniq = np.unique(labels[mask])
    result = dict(n_clusters=len(uniq), noise=(labels==-1).sum())
    if len(uniq) >= 2 and mask.sum() > len(uniq):
        try:
            result["silhouette"] = float(silhouette_score(X[mask], labels[mask], metric="euclidean"))
        except Exception:
            result["silhouette"] = np.nan
        try:
            result["calinski_harabasz"] = float(calinski_harabasz_score(X[mask], labels[mask]))
        except Exception:
            result["calinski_harabasz"] = np.nan
        try:
            result["davies_bouldin"] = float(davies_bouldin_score(X[mask], labels[mask]))
        except Exception:
            result["davies_bouldin"] = np.nan
    else:
        result.update(silhouette=np.nan, calinski_harabasz=np.nan, davies_bouldin=np.nan)
    return result

def kmeans_sweep(X, k_list=(4,6,8,10,12), random_state=42):
    rows = []
    for k in k_list:
        km = KMeans(n_clusters=k, n_init="auto", random_state=random_state)
        labels = km.fit_predict(X)
        m = evaluate_clustering(X, labels)
        m.update(alg="kmeans++", k=k)
        rows.append(m)
    return pd.DataFrame(rows).sort_values(by=["silhouette","calinski_harabasz"], ascending=[False, False])

def dbscan_sweep(X, eps_list=(0.5, 1.0, 1.5, 2.0, 3.0), min_samples_list=(5,10,20)):
    rows = []
    for eps in eps_list:
        for ms in min_samples_list:
            db = DBSCAN(eps=eps, min_samples=ms, n_jobs=-1)
            labels = db.fit_predict(X)
            m = evaluate_clustering(X, labels)
            m.update(alg="dbscan", eps=eps, min_samples=ms)
            rows.append(m)
    # Rank by silhouette (desc), then DB index (asc), noise (asc)
    df = pd.DataFrame(rows)
    return df.sort_values(by=["silhouette", "davies_bouldin", "noise"],
                          ascending=[False, True, True])

def louvain_on_graph(G, X=None, nodes_order=None, random_state=42):
    """
    Runs Louvain on graph G. 
    - nodes_order: order to return labels in (use the same order as your embeddings)
    - X: optional feature matrix to compute silhouette/CH/DB for Louvain labels
    """
    if nodes_order is None:
        nodes_order = list(G.nodes())

    # Compute partition (dict: node -> community)
    part = community_louvain.best_partition(G, random_state=random_state)
    # Align labels to the provided node order (important!)
    labels = np.array([part[n] for n in nodes_order])

    # Modularity on the graph partition
    Q = community_louvain.modularity(part, G)
    res = {"alg":"louvain", "modularity": float(Q), "n_clusters": int(len(set(labels)))}

    # Optional: evaluate these labels in your embedding space X
    if X is not None:
        from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
        try:
            res["silhouette"] = float(silhouette_score(X, labels))
        except Exception:
            res["silhouette"] = np.nan
        try:
            res["calinski_harabasz"] = float(calinski_harabasz_score(X, labels))
        except Exception:
            res["calinski_harabasz"] = np.nan
        try:
            res["davies_bouldin"] = float(davies_bouldin_score(X, labels))
        except Exception:
            res["davies_bouldin"] = np.nan

    return labels, res


In [12]:
# ---- YEAST ----
print("=== YEAST: K-MEANS++ SWEEP ===")
km_y = kmeans_sweep(X_y, k_list=(4,6,8,10,12,16))
display(km_y.head(10))

print("\n=== YEAST: DBSCAN SWEEP ===")
db_y = dbscan_sweep(X_y, eps_list=(0.5, 0.8, 1.0, 1.5, 2.0), min_samples_list=(5,10,20,50))
display(db_y.head(10))

print("\n=== YEAST: LOUVAIN ===")
labels_louvain_y, louvain_y = louvain_on_graph(G_yeast, X=X_y)
pd.DataFrame([louvain_y])


=== YEAST: K-MEANS++ SWEEP ===


,n_clusters,noise,silhouette,calinski_harabasz,davies_bouldin,alg,k
1,6,0,0.90,103.69,0.06,kmeans++,6
0,4,0,0.90,96.93,0.06,kmeans++,4
3,10,0,0.80,117.16,0.07,kmeans++,10
2,8,0,0.80,113.71,0.08,kmeans++,8
5,16,0,0.79,128.35,0.07,kmeans++,16
4,12,0,0.79,121.62,0.07,kmeans++,12



=== YEAST: DBSCAN SWEEP ===


,n_clusters,noise,silhouette,calinski_harabasz,davies_bouldin,alg,eps,min_samples
2,2,3927,0.95,23004.56,0.04,dbscan,0.50,20
1,5,3740,0.95,22113.78,0.03,dbscan,0.50,10
6,2,2220,0.92,10182.61,0.05,dbscan,0.80,20
5,5,2107,0.92,10735.08,0.04,dbscan,0.80,10
10,2,1650,0.91,7643.66,0.06,dbscan,1.00,20
9,5,1558,0.91,8143.27,0.05,dbscan,1.00,10
4,13,1971,0.90,7570.37,0.04,dbscan,0.80,5
14,2,1031,0.89,5041.95,0.07,dbscan,1.50,20
13,5,955,0.89,5361.98,0.06,dbscan,1.50,10
17,5,711,0.88,4251.11,0.06,dbscan,2.00,10



=== YEAST: LOUVAIN ===


,alg,modularity,n_clusters,silhouette,calinski_harabasz,davies_bouldin
0,louvain,0.17,10,-0.21,34.91,4.23


What each metric means (quick refresher)
silhouette (↑ better): how compact & separated clusters are (range ≈ −1 to 1).
calinski_harabasz / CH (↑ better): higher = tighter clusters, clearer separation.
davies_bouldin / DB (↓ better): lower = better separation.
n_clusters: number of clusters found.
noise (DBSCAN only): points labeled −1 (not assigned to any cluster).
modularity (↑ better) (Louvain only): community structure strength on the graph (not embeddings). Rough rule: ~0.3+ often indicates meaningful modularity; 0.1–0.2 is weak.

Yeast:
1) K-Means: 
Top rows show:
silhouette ≈ 0.80–0.90 across k = 4…16 (very strong).
DB ≈ 0.06–0.08 (low = good).
CH ≈ 97–128 (higher is better; consistent and healthy).
Interpretation:
The SVD embeddings for Yeast are very well separated into compact, roughly spherical groups. K-Means++ is a great fit here.
The best silhouettes are at k=4 and k=6 (≈0.90).
Your choice between k=4 and k=6 can be based on interpretability (fewer vs. more granular clusters). CH is slightly higher for k=16, but silhouette drops a bit — that often means you’re over-fragmenting.
2) DBSCAN (Yeast)
Top rows look amazing at first glance:
silhouette up to 0.95, DB as low as 0.03, CH huge…
…but check the noise column:
For the top settings, noise = 3927, 3740, 2220… out of 6526 nodes → 33–60% of nodes are being rejected as noise.
Interpretation (important nuance):
DBSCAN’s silhouette is computed only on the points it clustered (excluding noise). So those 0.95 silhouettes are high because the algorithm is keeping only the easiest/core points and discarding a large fraction as noise. That’s not necessarily “better clustering” overall.
3) Louvain (Yeast)
modularity = 0.17 (low/weak community structure on the graph).
n_clusters = 10
silhouette = −0.21 (negative) when you project those Louvain labels into SVD space.
Interpretation:
Louvain is optimizing modularity on the graph (not the SVD space). Here, the graph’s community structure is weak (0.17), which is consistent with the negative silhouette you see when you check those communities in the SVD space.
This doesn’t mean Louvain is “wrong”; it means the PPI graph (as loaded) doesn’t exhibit strong modular partitions under default resolution, or that the SVD embedding space doesn’t align with Louvain’s cut.
What to write:
“Louvain yields low modularity (0.17) and a negative silhouette in SVD space, indicating weak modular structure for Yeast under default settings. This contrasts with K-Means++ on SVD embeddings, suggesting the SVD feature space separates nodes differently than the modularity objective on the raw graph.”

In [13]:
# ---- HUMAN ----
print("=== HUMAN: K-MEANS++ SWEEP ===")
km_h = kmeans_sweep(X_h, k_list=(8,10,12,16,20,24))
display(km_h.head(10))

print("\n=== HUMAN: DBSCAN SWEEP ===")
db_h = dbscan_sweep(X_h, eps_list=(0.8, 1.0, 1.5, 2.0, 3.0), min_samples_list=(5,10,20,50))
display(db_h.head(10))

print("\n=== HUMAN: LOUVAIN ===")
labels_louvain_h, louvain_h = louvain_on_graph(G_human, X=X_h)
pd.DataFrame([louvain_h])


=== HUMAN: K-MEANS++ SWEEP ===


,n_clusters,noise,silhouette,calinski_harabasz,davies_bouldin,alg,k
5,24,0,0.41,437.81,0.43,kmeans++,24
4,20,0,0.39,405.67,0.68,kmeans++,20
3,16,0,0.37,382.24,0.72,kmeans++,16
2,12,0,0.34,361.49,0.44,kmeans++,12
1,10,0,0.33,345.22,0.46,kmeans++,10
0,8,0,0.31,328.78,0.49,kmeans++,8



=== HUMAN: DBSCAN SWEEP ===


,n_clusters,noise,silhouette,calinski_harabasz,davies_bouldin,alg,eps,min_samples
3,8,7866,0.87,33434.19,0.06,dbscan,0.80,50
7,8,6570,0.85,22220.91,0.07,dbscan,1.00,50
11,8,4543,0.79,11472.89,0.09,dbscan,1.50,50
10,23,3712,0.77,8675.57,0.08,dbscan,1.50,20
15,8,3669,0.76,8589.95,0.11,dbscan,2.00,50
14,23,2866,0.74,6402.87,0.09,dbscan,2.00,20
13,50,2313,0.73,4775.78,0.07,dbscan,2.00,10
2,27,6636,0.72,18090.67,0.09,dbscan,0.80,20
19,8,2608,0.70,5099.58,0.13,dbscan,3.00,50
9,53,3066,0.70,6030.22,0.09,dbscan,1.50,10



=== HUMAN: LOUVAIN ===


,alg,modularity,n_clusters,silhouette,calinski_harabasz,davies_bouldin
0,louvain,0.40,47,-0.61,22.77,5.71


1) K-MEANS++ (Human)
Top of your sweep:
k = 24 → silhouette 0.41, CH ≈ 438, DB ≈ 0.43
Smaller k steadily reduces silhouette (≈0.31–0.41 overall)
Interpretation:
Human PPI in SVD space does not form tight, spherical blobs. Even with many clusters (k=24), separation is only moderate. This tells you that Human’s structure is complex/overlapping, and K-Means++ is only partially capturing it.

2) DBSCAN (Human)
Top rows look great at first glance:
Silhouette up to 0.87, DB as low as 0.06.
…but check noise: thousands of nodes are being labeled −1 (not clustered). Given Human has ~21.6k nodes, those noise counts (e.g., 7–8k, 6–7k, 3–4k) mean a large fraction of nodes are being thrown away to get that high silhouette.
Interpretation:
DBSCAN is locking onto dense cores and dropping lots of points as noise. That boosts silhouette for the kept points, but it reduces coverage of the dataset. It also shows strong parameter sensitivity (eps, min_samples). With smaller min_samples or larger eps, you’ll reduce noise but silhouette will drop.

3) Louvain (Human)
Modularity = 0.40, n_clusters = 47
Silhouette in SVD space = −0.61 (negative), CH ≈ 22.8, DB ≈ 5.7
Interpretation:
0.40 modularity is meaningful community structure on the graph (this metric is computed on edges, not on the SVD coordinates).
The negative silhouette simply means those graph-communities don’t align as compact blobs in the SVD-projected space (which is common: modularity partitions need not be Euclidean-separable after linear projection).
Bottom line: Louvain is capturing network communities the embedding doesn’t separate well; that’s expected on a large, heterogeneous interactome.